In [267]:
import pandas as pd 
import numpy as np
import datetime 


In [268]:
df = pd.read_csv("G:/amazon-india-apparel-sales-analysis/Data/Amazon Sale Report.csv")

C:\Users\Hp\AppData\Local\Temp\ipykernel_12776\2462417131.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("G:/amazon-india-apparel-sales-analysis/Data/Amazon Sale Report.csv")


In [269]:
df_clean = df.copy()

In [270]:
df_clean.columns = (df_clean.columns
                    .str.strip()
                    .str.lower()
                    .str.replace("-", "_")
                    .str.replace(" ", "_"))

In [271]:
df_clean = df_clean.drop(columns=["index","sku", "asin", "ship_postal_code","unnamed:_22","promotion_ids","ship_city"])

In [273]:
df_clean = df_clean.drop_duplicates(subset="order_id", keep="first")

In [274]:
for col in ["status","fulfilment","sales_channel","ship_service_level","category","courier_status","ship_state"]:
    df_clean[col]= df_clean[col].str.strip().str.title()

In [275]:
df_clean["courier_status"] = df_clean["courier_status"].fillna("Not Available")
df_clean["currency"] = df_clean["currency"].fillna("INR")
df_clean["courier_status"] = df_clean["courier_status"].fillna("Not Available")


In [276]:
# average of real sales only (not cancelled, amount above 0)
avg = df_clean[(df_clean["status"] != "Cancelled") & (df_clean["amount"] > 0)]["amount"].mean()

# 229 non-cancelled nulls -> average (before the 0 fill)
mask = df_clean["amount"].isna() & (df_clean["status"] != "Cancelled")

#replacing the particular those rows with the column average
df_clean.loc[mask,"amount"] = avg

df_clean["amount"] = df_clean["amount"].fillna(0)

In [277]:
df_clean["revenue"] = df_clean["amount"].where(df_clean["status"] != "Cancelled", 0).round()

In [278]:
df_clean["date"] = pd.to_datetime(df_clean["date"], format= "%m-%d-%y")

In [279]:
df_clean["month"] = df_clean["date"].dt.month_name()
df_clean["month_num"] = df_clean["date"].dt.month
df_clean["day_name"] = df_clean["date"].dt.day_name()

In [280]:
state_map = {
    "Rajshthan": "Rajasthan", "Rajsthan": "Rajasthan", "Rj": "Rajasthan",
    "Pb": "Punjab", "Punjab/Mohali/Zirakpur": "Punjab",
    "New Delhi": "Delhi", "Orissa": "Odisha", "Pondicherry": "Puducherry",
    "Ar": "Arunachal Pradesh", "Nl": "Nagaland",
}
df_clean["ship_state"] = df_clean["ship_state"].replace(state_map)

for col in ["ship_state", "ship_country"]:
    df_clean[col] = df_clean[col].fillna("Unknown") 

In [281]:
df_clean["fulfilled_by"] = df_clean["fulfilled_by"].fillna("Not Applicable")

In [282]:
for col in ["order_id","status", "fulfilment", "sales_channel", "ship_service_level","style", "category", "size", "courier_status", "currency","ship_state", "ship_country", "fulfilled_by", "month", "day_name"]:

    df_clean[col] = df_clean[col].astype("string")

df_clean["qty"] = df_clean["qty"].astype("int64")
df_clean["month_num"] = df_clean["month_num"].astype("int64")
df_clean["amount"] = df_clean["amount"].astype("float64")
df_clean["revenue"] = df_clean["revenue"].astype("float64")

In [283]:
df_clean.isna().sum()

order_id              0
date                  0
status                0
fulfilment            0
sales_channel         0
ship_service_level    0
style                 0
category              0
size                  0
courier_status        0
qty                   0
currency              0
amount                0
ship_state            0
ship_country          0
b2b                   0
fulfilled_by          0
revenue               0
month                 0
month_num             0
day_name              0
dtype: int64

In [284]:
df_clean.nunique()

order_id              120378
date                      91
status                    13
fulfilment                 2
sales_channel              2
ship_service_level         2
style                   1358
category                   9
size                      11
courier_status             4
qty                        9
currency                   1
amount                  1379
ship_state                38
ship_country               2
b2b                        2
fulfilled_by               2
revenue                  849
month                      4
month_num                  4
day_name                   7
dtype: int64

In [285]:
df_clean.shape

(120378, 21)

In [286]:
df_clean.dtypes

order_id              string[python]
date                  datetime64[ns]
status                string[python]
fulfilment            string[python]
sales_channel         string[python]
ship_service_level    string[python]
style                 string[python]
category              string[python]
size                  string[python]
courier_status        string[python]
qty                            int64
currency              string[python]
amount                       float64
ship_state            string[python]
ship_country          string[python]
b2b                             bool
fulfilled_by          string[python]
revenue                      float64
month                 string[python]
month_num                      int64
day_name              string[python]
dtype: object

In [287]:
df_clean.to_csv("G:/amazon-india-apparel-sales-analysis/Data/amazon_sales_clean.csv",index=False)